Explore column discovery and mapping updates. The example uses a mock SQL client and does not contact a database.


In [ ]:
from setup.nbs.inspect_tools import show, show_methods
from pair_recon.core.metadata.mapping import discover_columns, discover_relations
from pair_recon.core.metadata.mapping import discover_oracle_columns, discover_hadoop_columns, discover_csv_columns
from pair_recon.core.metadata.mapping import discover_databricks_columns, discover_databricks_partition
from pair_recon.core.metadata.mapping import automatch_columns, get_date_column_type, upsert_column_mappings

show(discover_columns, discover_relations)
show(discover_oracle_columns, discover_hadoop_columns, discover_csv_columns)
show(discover_databricks_columns, discover_databricks_partition)
show(automatch_columns, get_date_column_type, upsert_column_mappings)


## Replace or patch mappings

`overwrite=True` replaces only mappings for the selected pair, preserving all other pairs. An empty overwrite clears only the selected pair. `overwrite=False` patches the selected pair using MERGE. This example only records SQL calls.


In [ ]:
from unittest.mock import Mock
import json

mapping = [{"left_column": "ORDER_ID", "right_column": "order_id", "data_type": "numeric"}]
for overwrite in (True, False):
    sql = Mock()
    result = upsert_column_mappings(sql, "demo.recon.pair_recon_column_map", "ORDERS", mapping, overwrite=overwrite)
    statement, parameters = sql.execute.call_args.args
    print("input:", mapping, "overwrite:", overwrite)
    print("result:", result)
    print("SQL:", statement.splitlines()[0])
    print("SQL input:", json.loads(parameters["mappings_json"]))

sql = Mock()
result = upsert_column_mappings(sql, "demo.recon.pair_recon_column_map", "ORDERS", [], overwrite=False)
print("empty patch:", result["rows"], "SQL calls:", sql.execute.call_count)


## Toy example

Run this cell locally to inspect the concrete inputs and outputs.


In [ ]:
left = ["ORDER_ID", "AMOUNT", "LEFT_ONLY"]
right = ["order_id", "AMOUNT", "RIGHT_ONLY"]
print("left:", left, "right:", right)
print("matched:", automatch_columns(left, right))
